# 数据更新脚本 v2 - 掘金数据源

使用掘金(GM)数据源更新股票数据，替代Tushare数据源

数据存储目录：
- `gm_stock_all_data` - 日线基础数据（包含行情、市值、复权因子等）

In [1]:
import sys
DATA_ROOT_DIR = r'E:\working\stock_data'
sys.path.append("C://Users/20561/Desktop/策略")

import polars as pl
import datetime
import pandas as pd
import os
from my_utils.fun import *
from my_utils.stock_api import *
from my_utils.mapping import *

# 初始化日志
logging = get_logger(log_file='log/数据更新v2.log', inherit=False)

# 初始化API
api = stock_api()

print(f"数据根目录: {DATA_ROOT_DIR}")
print(f"当前日期: {datetime.date.today()}")

数据根目录: E:\working\stock_data
当前日期: 2026-04-12


## 读取现有数据，获取最新日期

In [2]:
# 尝试读取现有掘金数据
gm_data_dir = os.path.join(DATA_ROOT_DIR, 'gm_stock_all_data')

if os.path.exists(gm_data_dir):
    try:
        existing_data = read_day_data(
            start_date=datetime.datetime(2020, 1, 1),
            end_date=datetime.datetime.today(),
            file_path='gm_stock_all_data'
        )
        latest_date = existing_data.select(pl.col("trading_date").max()).item()
        print(f"✓ 现有数据最新日期: {latest_date}")
        print(f"  总记录数: {existing_data.height}")
        print(f"  字段数: {len(existing_data.columns)}")
        print(f"\n字段列表:")
        for i, col in enumerate(existing_data.columns, 1):
            print(f"  {i}. {col}")
    except Exception as e:
        print(f"⚠ 读取现有数据失败: {e}")
        existing_data = None
        latest_date = datetime.date(2020, 1, 1)
else:
    print(f"ℹ 数据目录不存在，将创建新数据: {gm_data_dir}")
    existing_data = None
    latest_date = datetime.date(2020, 1, 1)
    os.makedirs(gm_data_dir, exist_ok=True)

✓ 现有数据最新日期: 2026-04-10
  总记录数: 1598842
  字段数: 18

字段列表:
  1. code
  2. name
  3. trading_date
  4. open
  5. high
  6. low
  7. close
  8. pre_close
  9. volume
  10. amount
  11. limit_up
  12. limit_down
  13. is_st
  14. is_suspended
  15. adj_factor
  16. turnover_rate
  17. total_mv
  18. mv_A_free_float


## 更新日线基础数据

In [3]:
def update_day_data_gm(day_data, save_dir='gm_stock_all_data', mode='update'):
    """
    更新日线基础数据到Parquet分区文件
    
    参数:
        day_data: polars DataFrame，包含日线数据
        save_dir: 保存目录名称
        mode: 更新模式，'insert'表示增量更新，'update'表示全量更新
    """
    save_path = os.path.join(DATA_ROOT_DIR, save_dir)
    
    # 创建目录（如果不存在）
    if not os.path.exists(save_path):
        os.makedirs(save_path)
        print(f"创建目录: {save_path}")
    
    # 获取已有日期列表
    existing_dates = []
    for item in os.listdir(save_path):
        if item.startswith("trading_date="):
            date_str = item.split("=")[1]
            existing_dates.append(date_str)
    
    if mode == 'insert' and existing_dates:
        # 增量更新：只保留大于最新日期的数据
        start_date_str = max(existing_dates)
        start_date = datetime.datetime.strptime(start_date_str, '%Y-%m-%d').date()
        new_data = day_data.filter(pl.col("trading_date") > start_date)
        print(f"增量更新模式: 保留 > {start_date} 的数据")
    else:
        new_data = day_data
        print(f"全量更新模式: 更新全部数据")
    
    if new_data.is_empty():
        print("没有新数据需要更新")
        return
    
    # 获取已有数据的schema并转换
    existing_schema = get_parquet_dir_schema(save_path)
    
    if existing_schema:
        # 强制转换新数据的列类型以匹配已有schema
        convert_exprs = []
        for col, dtype in existing_schema.items():
            if col in new_data.columns:
                convert_exprs.append(pl.col(col).cast(dtype).alias(col))
        
        if convert_exprs:
            new_data = new_data.select(convert_exprs)
        
        # 确保所有schema中的列都存在于新数据中
        missing_cols = [col for col in existing_schema.keys() if col not in new_data.columns]
        if missing_cols:
            print(f"警告: 新数据缺少以下列，已自动添加空值列: {missing_cols}")
            for col in missing_cols:
                new_data = new_data.with_column(pl.lit(None).cast(existing_schema[col]).alias(col))
    else:
        print("目录中没有数据，直接添加新数据")
    
    # 排序并保存
    new_data = new_data.sort(['trading_date', 'code'])
    print(f"准备更新日线数据，共 {new_data.height} 条记录")
    new_data.write_parquet(save_path, partition_by=['trading_date'])
    print(f"✓ 数据已保存到: {save_path}")

# 测试函数
print("update_day_data_gm 函数已定义")

update_day_data_gm 函数已定义


## 获取并更新数据

In [ ]:
# 设置更新参数
end_date = datetime.date.today()

# 如果是首次运行，可以设置一个起始日期
# 否则从现有数据的最新日期开始
if existing_data is not None:
    # 从最新日期的下一天开始
    start_date = latest_date + datetime.timedelta(days=1)
else:
    # 首次运行，设置起始日期（可根据需要调整）
    start_date = datetime.date(2021, 1, 1)  # 修改为你需要的起始日期


print(f"更新日期范围: {start_date} ~ {end_date}")

# 检查是否需要更新
if start_date > end_date:
    print("数据已是最新，无需更新")
else:
    print(f"需要获取 { (end_date - start_date).days + 1 } 天的数据")

更新日期范围: 2020-01-01 ~ 2021-01-01
需要获取 367 天的数据


In [12]:
# 使用掘金API批量获取数据
# 注意：这可能需要较长时间，取决于日期范围

print("开始获取掘金数据...")
print(f"起始日期: {start_date}")
print(f"结束日期: {end_date}")
print("\n正在获取数据，请稍候...")

# 调用批量获取接口
day_data_df = api.gm_get_daily_data_multi_dates(
    start_date=str(start_date),
    end_date=str(end_date)
)

if day_data_df is not None and not day_data_df.empty:
    print(f"\n✓ 数据获取成功！共 {len(day_data_df)} 条记录")
    print(f"  交易日数量: {day_data_df['trading_date'].nunique()}")
    print(f"  股票数量: {day_data_df['code'].nunique()}")
    
    # 显示数据预览
    print("\n数据预览:")
    print(day_data_df.head())
    
    # 转换为Polars DataFrame
    day_data_pl = pl.from_pandas(day_data_df)
    
    # 确保trading_date是date类型
    if day_data_pl['trading_date'].dtype != pl.Date:
        day_data_pl = day_data_pl.with_columns(
            pl.col('trading_date').cast(pl.Date).alias('trading_date')
        )
    
    print("\n✓ 数据已转换为Polars格式")
else:
    print("✗ 数据获取失败或返回空数据")
    day_data_pl = None

准备获取 263 个交易日的数据: 2020-01-01 至 2021-01-01
开始获取 2020-01-01 的掘金全市场数据...
  步骤1: 调用get_symbols获取基础信息...


开始获取掘金数据...
起始日期: 2020-01-01
结束日期: 2021-01-01

正在获取数据，请稍候...


2020-01-01 get_symbols返回空数据
✗ 2020-01-01 无数据
开始获取 2020-01-02 的掘金全市场数据...
  步骤1: 调用get_symbols获取基础信息...
  get_symbols返回 3835 条记录
  过滤后剩余 3835 只股票(已剔除未上市/已退市)
  步骤2: 调用history批量获取数据...
  history批量返回 3835 条记录
  步骤3: 调用stk_get_daily_mktvalue_pt获取市值数据...
  stk_get_daily_mktvalue_pt返回 3835 条记录
  步骤4: 合并数据...
  数据合并完成,最终返回 3835 条记录, 18 个字段
✓ 2020-01-02 数据获取成功: 3835 条
开始获取 2020-01-03 的掘金全市场数据...
  步骤1: 调用get_symbols获取基础信息...
  get_symbols返回 3835 条记录
  过滤后剩余 3835 只股票(已剔除未上市/已退市)
  步骤2: 调用history批量获取数据...
  history批量返回 3835 条记录
  步骤3: 调用stk_get_daily_mktvalue_pt获取市值数据...
  stk_get_daily_mktvalue_pt返回 3835 条记录
  步骤4: 合并数据...
  数据合并完成,最终返回 3835 条记录, 18 个字段
✓ 2020-01-03 数据获取成功: 3835 条
开始获取 2020-01-06 的掘金全市场数据...
  步骤1: 调用get_symbols获取基础信息...
  get_symbols返回 3838 条记录
  过滤后剩余 3838 只股票(已剔除未上市/已退市)
  步骤2: 调用history批量获取数据...
  history批量返回 3838 条记录
  步骤3: 调用stk_get_daily_mktvalue_pt获取市值数据...
  stk_get_daily_mktvalue_pt返回 3838 条记录
  步骤4: 合并数据...
  数据合并完成,最终返回 3838 条记录, 18 个字段
✓ 2020-01-06 数据获取成功: 3838 条
开


✓ 数据获取成功！共 968477 条记录
  交易日数量: 243
  股票数量: 4251

数据预览:
          code  name trading_date   open   high    low  close  pre_close  \
0  SHSE.600000  浦发银行   2020-01-02  12.47  12.64  12.45  12.47      12.37   
1  SHSE.600004  白云机场   2020-01-02  17.56  17.62  17.40  17.52      17.45   
2  SHSE.600006  东风股份   2020-01-02   4.62   4.67   4.60   4.65       4.58   
3  SHSE.600007  中国国贸   2020-01-02  17.62  17.66  17.44  17.55      17.50   
4  SHSE.600008  首创环保   2020-01-02   3.31   3.34   3.30   3.33       3.29   

     volume       amount  limit_up  limit_down  is_st  is_suspended  \
0  51629079  647446166.0     13.61       11.13  False         False   
1  17792404  311394028.0     19.20       15.71  False         False   
2  13212901   61341914.0      5.04        4.12  False         False   
3   1621727   28474105.0     19.25       15.75  False         False   
4  21498259   71460782.0      3.62        2.96  False         False   

   adj_factor  turnover_rate        total_mv  mv_A_free_floa

In [9]:
# 执行更新
if day_data_pl is not None and not day_data_pl.is_empty():
    print("\n开始更新数据到本地...")
    update_day_data_gm(day_data_pl, save_dir='gm_stock_all_data', mode='update')
    print("\n✓ 数据更新完成！")
else:
    print("没有数据需要更新")


开始更新数据到本地...
全量更新模式: 更新全部数据
成功从 E:\working\stock_data\gm_stock_all_data\trading_date=2025-01-02\0.parquet 读取schema
准备更新日线数据，共 4731579 条记录
✓ 数据已保存到: E:\working\stock_data\gm_stock_all_data

✓ 数据更新完成！


## 验证更新结果

In [10]:
# 读取更新后的数据验证
try:
    updated_data = read_day_data(
        start_date=datetime.datetime(2020, 1, 1),
        end_date=datetime.datetime.today(),
        file_path='gm_stock_all_data'
    )
    
    print("✓ 数据读取成功")
    print(f"\n更新后统计:")
    print(f"  总记录数: {updated_data.height}")
    print(f"  最新日期: {updated_data.select(pl.col('trading_date').max()).item()}")
    print(f"  最早日期: {updated_data.select(pl.col('trading_date').min()).item()}")
    print(f"  交易日数量: {updated_data.select(pl.col('trading_date').n_unique()).item()}")
    print(f"  股票数量: {updated_data.select(pl.col('code').n_unique()).item()}")
    
    # 显示最近几天的数据统计
    print("\n最近5个交易日数据量:")
    recent_stats = (
        updated_data
        .group_by('trading_date')
        .agg(pl.count().alias('count'))
        .sort('trading_date', descending=True)
        .head(5)
    )
    print(recent_stats.to_pandas().to_string(index=False))
    
except Exception as e:
    print(f"✗ 读取数据失败: {e}")

✓ 数据读取成功

更新后统计:
  总记录数: 6330421
  最新日期: 2026-04-10
  最早日期: 2021-01-04
  交易日数量: 1275
  股票数量: 5481

最近5个交易日数据量:


C:\Users\20561\AppData\Local\Temp\ipykernel_50260\1913028384.py:22: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias('count'))


trading_date  count
  2026-04-10   5268
  2026-04-09   5265
  2026-04-08   5264
  2026-04-07   5259
  2026-04-03   5261


## 数据对比（与Tushare数据）

可选：对比掘金数据和Tushare数据的差异

In [8]:
# 读取Tushare数据进行对比（可选）
try:
    ts_data = read_day_data(
        start_date=datetime.datetime(2025, 4, 1),
        end_date=datetime.datetime.today(),
        file_path='ts_stock_all_data'
    )
    
    print("Tushare数据:")
    print(f"  总记录数: {ts_data.height}")
    print(f"  字段数: {len(ts_data.columns)}")
    
    print("\n掘金数据:")
    print(f"  总记录数: {updated_data.height}")
    print(f"  字段数: {len(updated_data.columns)}")
    
    print("\n字段对比:")
    ts_cols = set(ts_data.columns)
    gm_cols = set(updated_data.columns)
    
    print(f"  Tushare特有字段: {ts_cols - gm_cols}")
    print(f"  掘金特有字段: {gm_cols - ts_cols}")
    print(f"  共有字段: {ts_cols & gm_cols}")
    
except Exception as e:
    print(f"对比失败: {e}")

Tushare数据:
  总记录数: 1268610
  字段数: 33

掘金数据:
  总记录数: 1598842
  字段数: 18

字段对比:
  Tushare特有字段: {'change', 'pct', 'float_share', 'buying', 'strength', 'pe', 'attack', 'total_share', 'turn_over', 'avg_turnover', 'industry', 'avg_price', 'vol_ratio', 'swing', 'activity', 'float_mv', 'type', 'area', 'selling', 'type_name'}
  掘金特有字段: {'is_suspended', 'is_st', 'adj_factor', 'mv_A_free_float', 'turnover_rate'}
  共有字段: {'limit_down', 'high', 'limit_up', 'trading_date', 'open', 'low', 'amount', 'name', 'volume', 'total_mv', 'code', 'close', 'pre_close'}


## 单次更新（用于补充特定日期数据）

如果需要更新特定日期，可以使用以下代码：

In [ ]:
# 更新特定日期（示例）
# specific_date = '2025-04-10'
# single_day_df = api.gm_get_daily_data(specific_date)
# if single_day_df is not None:
#     single_day_pl = pl.from_pandas(single_day_df)
#     single_day_pl = single_day_pl.with_columns(
#         pl.col('trading_date').cast(pl.Date).alias('trading_date')
#     )
#     update_day_data_gm(single_day_pl, save_dir='gm_stock_all_data', mode='insert')
#     print(f"✓ {specific_date} 数据更新完成")